# Regresión Logística

---
Nombre: **Alan Jesús Hernández Soto** \
Carrera: **Ingeniería Financiera** \
Materia: **Laboratorio de Aprendizaje Estadístico** \
Tipo de Trabajo: **Clase**

---

## Carga de datos y librerias

In [129]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LogisticRegression

df = pd.read_csv('Datasets/Heart Prediction Quantum Dataset.csv')
df

,Age,Gender,BloodPressure,Cholesterol,HeartRate,HeartDisease
0,68,Hombre,105.00,191,107,1
1,58,Mujer,97.00,249,89,0
2,44,Mujer,93.00,190,82,1
3,72,Hombre,93.00,183,101,1
4,37,Mujer,145.00,166,103,1
...,...,...,...,...,...,...
495,34,Mujer,126.00,292,116,0
496,41,Mujer,164.00,248,114,0
497,45,Hombre,159.00,175,75,0
498,55,Mujer,107.00,157,101,1


## Limpieza de Datos e ingeniería de características

In [130]:
df['BloodPressure'] = (df['BloodPressure'].str.replace(',', '.').astype(float))

gend_min = df['Gender'].str.lower()
gend_min = gend_min.str.rstrip(' ')
gend_min = gend_min.str.rstrip('{')
gend_min = gend_min.str.rstrip('}')
gend_min[gend_min == 'ombre'] = 'hombre'
gend_min.unique()

gend_min[gend_min == 'hombre'] = 1
gend_min[gend_min == 'mujer'] = 0

df['Gender'] = gend_min.astype(int)

df.drop_duplicates(inplace=True)

df.dropna(inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Age            500 non-null    int64  
 1   Gender         500 non-null    int64  
 2   BloodPressure  500 non-null    float64
 3   Cholesterol    500 non-null    int64  
 4   HeartRate      500 non-null    int64  
 5   HeartDisease   500 non-null    int64  
dtypes: float64(1), int64(5)
memory usage: 23.6 KB


## Regresión logística

La regresión se usa con la función `LogisticRegression()` de Sklearn, el proceso de definición, transformación y fitting de X, Y es igual que en regresión lineal.

In [131]:
lr = LogisticRegression()
X = df[['Age', 'Gender', 'BloodPressure', 'Cholesterol', 'HeartRate']].values
Y = df['HeartDisease'].values

lr.fit(X, Y)
lr.intercept_.flatten(), lr.coef_.flatten()

prob_y_1 = lr.predict_proba(X)[:, 1]
umb_03 = [prob_y_1 > 0.3]
umb_05 = [prob_y_1 > 0.5]
umb_08 = [prob_y_1 > 0.8]

TP_03 = np.sum((Y == 1) & (umb_03[0] == True))
FP_03 = np.sum((Y == 0) & (umb_03[0] == True))
TN_03 = np.sum((Y == 0) & (umb_03[0] == False))
FN_03 = np.sum((Y == 1) & (umb_03[0] == False))

TP_05 = np.sum((Y == 1) & (umb_05[0] == True))
FP_05 = np.sum((Y == 0) & (umb_05[0] == True))
TN_05 = np.sum((Y == 0) & (umb_05[0] == False))
FN_05 = np.sum((Y == 1) & (umb_05[0] == False))

TP_08 = np.sum((Y == 1) & (umb_08[0] == True))
FP_08 = np.sum((Y == 0) & (umb_08[0] == True))
TN_08 = np.sum((Y == 0) & (umb_08[0] == False))
FN_08 = np.sum((Y == 1) & (umb_08[0] == False))

## Métricas

In [132]:
accuracy_03 = (TP_03 + TN_03) / (TP_03 + FP_03 + TN_03 + FN_03)
recall_03 = TP_03 / (TP_03 + FN_03)
precision_03 = TP_03 / (TP_03 + FP_03)
specificity_03 = TN_03 / (TN_03 + FP_03)

accuracy_05 = (TP_05 + TN_05) / (TP_05 + FP_05 + TN_05 + FN_05)
recall_05 = TP_05 / (TP_05 + FN_05)
precision_05 = TP_05 / (TP_05 + FP_05)
specificity_05 = TN_05 / (TN_05 + FP_05)

accuracy_08 = (TP_08 + TN_08) / (TP_08 + FP_08 + TN_08 + FN_08)
recall_08 = TP_08 / (TP_08 + FN_08)
precision_08 = TP_08 / (TP_08 + FP_08)
specificity_08 = TN_08 / (TN_08 + FP_08)

In [133]:
df_metrics = pd.DataFrame({
    'Threshold': [0.3, 0.5, 0.8],
    'Accuracy': [accuracy_03, accuracy_05, accuracy_08],
    'Recall': [recall_03, recall_05, recall_08],
    'Precision': [precision_03, precision_05, precision_08],
    'Specificity': [specificity_03, specificity_05, specificity_08]
})

df_metrics

,Threshold,Accuracy,Recall,Precision,Specificity
0,0.3,0.708,0.936667,0.688725,0.365
1,0.5,0.730,0.793333,0.765273,0.635
2,0.8,0.636,0.446667,0.893333,0.920


A menor umbral el recall es mucho mayor ya que, dejamos pasar muchos mas positivos entre mas pequeño es el umbral pero esto recae en la precision ya que aqui tenemos en cuenta los falsos positivos que aceptamos. La relación con el Specifity es a la inversa a mayor umbral, el specifity es mucho mayor ya que sucede locontrario, dejamos pasar muchos negativos entre mas grande es el umbral. Es de mencionar que en terminos de accuracy el modelo de 0.5 es el mejor, ningun extremo.

## Variación de C (Penalización)

In [134]:
lr = LogisticRegression(C=1e-6)
X = df[['Age', 'Gender', 'BloodPressure', 'Cholesterol', 'HeartRate']].values
Y = df['HeartDisease'].values

lr.fit(X, Y)
lr.intercept_.flatten(), lr.coef_.flatten()


(array([1.31174549]),
 array([ 8.80812814e-04, -1.19458732e-06, -7.18407429e-04, -3.69815601e-03,
        -4.21058210e-04]))

In [135]:
lr = LogisticRegression(C=1e6)
X = df[['Age', 'Gender', 'BloodPressure', 'Cholesterol', 'HeartRate']].values
Y = df['HeartDisease'].values

lr.fit(X, Y)
lr.intercept_.flatten(), lr.coef_.flatten()

(array([7.07049453]),
 array([ 0.05036867, -0.02996471, -0.01585998, -0.02510933, -0.01787689]))

Si queremos mucho mas fuerza de penalización nuestro C debe ser muy pequeño por lo tanto hay una relación inversa entre el C y la fuerza de penalización.